In [2]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
import librosa
import tensorflow as tf
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [3]:
EMOTIONS        = ["angry", "happy", "sad"]
IMAGE_DIR       = "dataset"        # subfolders: angry/ happy/ sad/
AUDIO_DIR       = "dataset_voice"  # subfolders: angry/ happy/ sad/
IMAGE_EXTS      = (".jpg", ".jpeg", ".png")
AUDIO_EXTS      = (".wav", ".mp3", ".flac")
IMG_SIZE        = (48, 48)
SAMPLE_RATE     = 22050
DURATION        = 3.0
N_MFCC          = 13
PAIRS_PER_CLASS = 50    # how many image+audio pairs to sample per emotion
RANDOM_SEED     = 42
 
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cnn    = tf.keras.models.load_model("Full_model_AI.h5")
svm    = joblib.load("svm_model.pkl")
scaler = joblib.load("scaler.pkl")

In [4]:
# Feature extraction functions

def extract_audio_features(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target   = int(SAMPLE_RATE * DURATION)
    if len(audio) < target:
        audio = np.pad(audio, (0, target - len(audio)))
    mfccs       = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    pitch       = librosa.yin(audio, fmin=librosa.note_to_hz("C2"),
                               fmax=librosa.note_to_hz("C7"))
    energy      = librosa.feature.rms(y=audio)
    return np.append(np.mean(mfccs, axis=1), [np.mean(pitch), np.mean(energy)])
 
 
def predict_cnn(image_path):
    img  = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr  = tf.keras.utils.img_to_array(img) / 255.0
    arr  = np.expand_dims(arr, axis=0)
    return cnn.predict(arr, verbose=0)[0]   # shape (3,) softmax
 
 
def predict_svm(audio_path):
    feats  = extract_audio_features(audio_path)
    scaled = scaler.transform([feats])
    probs  = svm.predict_proba(scaled)[0]
    order  = list(svm.classes_)
    return np.array([probs[order.index(em)] for em in EMOTIONS])
 
 
# Build paired dataset
 
print(f"Sampling {PAIRS_PER_CLASS} pairs per emotion...")
 
cnn_probs_all = []   # (n_pairs, 3)
svm_probs_all = []   # (n_pairs, 3)
labels_all    = []   # (n_pairs,)  integer class index
 
for em_idx, emotion in enumerate(EMOTIONS):
    img_folder   = os.path.join(IMAGE_DIR, emotion)
    audio_folder = os.path.join(AUDIO_DIR, emotion)
 
    img_files   = [f for f in os.listdir(img_folder)   if f.endswith(IMAGE_EXTS)]
    audio_files = [f for f in os.listdir(audio_folder) if f.endswith(AUDIO_EXTS)]
 
    # Sample with replacement if not enough files
    n = PAIRS_PER_CLASS
    sampled_imgs   = random.choices(img_files,   k=n)
    sampled_audios = random.choices(audio_files, k=n)
 
    success = 0
    for img_f, aud_f in zip(sampled_imgs, sampled_audios):
        try:
            cp = predict_cnn(os.path.join(img_folder,   img_f))
            sp = predict_svm(os.path.join(audio_folder, aud_f))
            cnn_probs_all.append(cp)
            svm_probs_all.append(sp)
            labels_all.append(em_idx)
            success += 1
        except Exception as e:
            print(f"  [skip] {img_f} / {aud_f} — {e}")
 
    print(f"  {emotion}: {success} pairs collected")
 
cnn_probs_all = np.array(cnn_probs_all)   # (N, 3)
svm_probs_all = np.array(svm_probs_all)   # (N, 3)
labels_all    = np.array(labels_all)       # (N,)
N             = len(labels_all)
print(f"\nTotal pairs: {N}\n")

Sampling 50 pairs per emotion...
  angry: 50 pairs collected
  happy: 50 pairs collected
  sad: 50 pairs collected

Total pairs: 150



In [5]:
def neg_accuracy(w_cnn):
    w = float(np.clip(w_cnn, 0, 1))
    combined = w * cnn_probs_all + (1 - w) * svm_probs_all
    preds    = np.argmax(combined, axis=1)
    return -accuracy_score(labels_all, preds)
 
 
# Grid search first to find a good starting point
print("Grid searching over weights...")
grid     = np.linspace(0, 1, 101)
grid_acc = [-neg_accuracy(w) for w in grid]
best_grid_w   = grid[np.argmax(grid_acc)]
best_grid_acc = max(grid_acc)
print(f"  Best grid weight  — CNN: {best_grid_w:.2f} | SVM: {1-best_grid_w:.2f} | Acc: {best_grid_acc*100:.1f}%")
 
# Refine with scipy
result = minimize(neg_accuracy, x0=[best_grid_w],
                  method="L-BFGS-B", bounds=[(0.0, 1.0)],
                  options={"ftol": 1e-9, "gtol": 1e-7})
 
best_w_cnn = float(np.clip(result.x[0], 0, 1))
best_w_svm = 1.0 - best_w_cnn
best_acc   = -result.fun
 
print(f"\n{'─'*45}")
print(f"  Optimal CNN weight : {best_w_cnn:.4f}  ({best_w_cnn*100:.1f}%)")
print(f"  Optimal SVM weight : {best_w_svm:.4f}  ({best_w_svm*100:.1f}%)")
print(f"  Combined accuracy  : {best_acc*100:.1f}%")
print(f"{'─'*45}\n")
 
# Baselines for comparison
cnn_only_acc = accuracy_score(labels_all, np.argmax(cnn_probs_all, axis=1))
svm_only_acc = accuracy_score(labels_all, np.argmax(svm_probs_all, axis=1))
print(f"  CNN only accuracy  : {cnn_only_acc*100:.1f}%")
print(f"  SVM only accuracy  : {svm_only_acc*100:.1f}%")
 
# Save weights
joblib.dump({"w_cnn": best_w_cnn, "w_svm": best_w_svm}, "fusion_weights.pkl")
print("\nSaved → fusion_weights.pkl")

Grid searching over weights...
  Best grid weight  — CNN: 0.52 | SVM: 0.48 | Acc: 94.0%

─────────────────────────────────────────────
  Optimal CNN weight : 0.5200  (52.0%)
  Optimal SVM weight : 0.4800  (48.0%)
  Combined accuracy  : 94.0%
─────────────────────────────────────────────

  CNN only accuracy  : 88.0%
  SVM only accuracy  : 82.0%

Saved → fusion_weights.pkl
